# Simulador 13.1 — Programas de Refuerzo

**Capítulo 13: La Ley del Efecto y los Programas de Refuerzo**  
*Aprendizaje y Comportamiento Adaptable: Principios y Modelos*  
Arturo Bouzas · Laboratorio 25 · UNAM

---

Este simulador genera registros acumulativos para los cuatro programas básicos de refuerzo:
**Razón Variable (RV), Razón Fija (RF), Intervalo Variable (IV) e Intervalo Fijo (IF)**.

El panel inferior muestra la **función de retroalimentación del entorno**: la relación entre
la tasa de respuesta del organismo y la tasa de reforzamiento que el entorno devuelve
bajo cada tipo de programa.

> **Instrucción:** Ejecuta todas las celdas en orden (*Runtime → Run all* o *Ctrl+F9*).
> Luego ajusta los controles y presiona **▶ Simular**. Registra tu predicción *antes* de correr la simulación.


In [ ]:
#@title **Simulador 13.1** — Programas de Refuerzo

# ==============================================================================
# Simulador 13.1 — Programas de Refuerzo
# Capítulo 13: La Ley del Efecto y los Programas de Refuerzo
# Aprendizaje y Comportamiento Adaptable: Principios y Modelos
# ==============================================================================

# ——— Dependencias —————————————————————————————————————————————————————————————
# En Google Colab, ipywidgets ya suele estar instalado.
# Si ves un error, descomenta la línea siguiente y vuelve a ejecutar.
# !pip install ipywidgets --quiet
try:
    import ipywidgets
except ImportError:
    print("⚠ ipywidgets no encontrado. Ejecuta: !pip install ipywidgets --quiet")

# ——— Importaciones ————————————————————————————————————————————————————————————
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display, clear_output

matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.serif']  = ['Georgia', 'Palatino Linotype', 'DejaVu Serif']
matplotlib.rcParams['axes.spines.top']   = False
matplotlib.rcParams['axes.spines.right'] = False

# ── Paleta del libro ──────────────────────────────────────────────────────────
AZUL      = '#2C5282'
NARANJA   = '#C05621'
VERDE     = '#276749'
GRIS      = '#718096'
GRIS_MED  = '#A0AEC0'
GRIS_CLAR = '#EDF2F7'
BLANCO    = '#FFFFFF'
PANEL_BG  = '#EBF4FF'   # fondo del panel de controles (estilo cap. 11)
PANEL_BRD = '#2C5282'   # borde del panel de controles

COLOR_RV = AZUL
COLOR_RF = VERDE
COLOR_IV = GRIS
COLOR_IF = NARANJA


# ── Helpers HTML (estilo referencia cap. 11) ──────────────────────────────────
def _html_header_131() -> str:
    return (
        f'<div style="background-color:{AZUL};color:white;'
        f'font-family:Georgia,serif;font-size:14px;font-weight:bold;'
        f'padding:8px 14px;border-radius:6px 6px 0 0;letter-spacing:0.5px;">'
        f'&nbsp;Simulador 13.1 &mdash; Programas de Refuerzo'
        f'<br><span style="font-size:11px;font-weight:normal;">'
        f'Ajusta los parámetros y presiona <b>&#9654; Simular</b>. '
        f'Formula tu predicci&oacute;n <em>antes</em> de correr la simulaci&oacute;n.'
        f'</span></div>'
    )


def _html_sec_131(texto: str) -> str:
    return (
        f'<div style="color:{AZUL};font-family:Georgia,serif;'
        f'font-size:11px;font-weight:bold;text-transform:uppercase;'
        f'letter-spacing:1px;margin:8px 0 2px 4px;">{texto}</div>'
    )


# ──────────────────────────────────────────────────────────────────────────────
# Funciones de simulación
# Cada función modela el comportamiento de un organismo virtual bajo un
# programa de refuerzo y devuelve dos arrays: tiempos de respuesta
# (en segundos) y tiempos en que se entregó el reforzador.
# ──────────────────────────────────────────────────────────────────────────────

def simular_RV(n_medio, n_reforzadores=30, tasa_base=4.5, seed=42):
    """
    Razón Variable (RV-n).
    El organismo responde a tasa alta y constante. El número de respuestas
    requerido varía según una distribución geométrica con media n_medio
    (correcta para programas de razón variable en estado estable).
    IRTs ~ Exponencial(1/tasa_base): proceso de Poisson estacionario.
    Patrón esperado: pendiente alta, sostenida, sin pausas apreciables.
    """
    rng = np.random.default_rng(seed)
    tiempos_resp, tiempos_ref = [], []
    t = 0.0
    for _ in range(n_reforzadores):
        requisito = max(1, int(rng.geometric(1.0 / n_medio)))
        for _ in range(requisito):
            t += rng.exponential(1.0 / tasa_base)
            tiempos_resp.append(t)
        tiempos_ref.append(t)
        # Sin pausa post-reforzador bajo RV
    return np.array(tiempos_resp), np.array(tiempos_ref)


def simular_RF(n_fijo, n_reforzadores=30, tasa_base=4.5, seed=42):
    """
    Razón Fija (RF-n).
    Requisito exacto de n_fijo respuestas. Tras cada reforzador, el organismo
    hace una pausa proporcional al tamaño del requisito (≈ 0.45 · n / tasa_base),
    con variabilidad ±40% para aproximar la distribución empírica.
    Patrón esperado: pausas post-reforzador regulares + carreras a tasa alta.
    """
    rng = np.random.default_rng(seed)
    tiempos_resp, tiempos_ref = [], []
    t = 0.0
    dur_pausa_media = 0.45 * (n_fijo / tasa_base)
    for _ in range(n_reforzadores):
        # Pausa post-reforzador (con variabilidad ±40%)
        t += rng.uniform(0.6 * dur_pausa_media, 1.4 * dur_pausa_media)
        # Carrera hasta completar el requisito
        for _ in range(n_fijo):
            t += rng.exponential(1.0 / tasa_base)
            tiempos_resp.append(t)
        tiempos_ref.append(t)
    return np.array(tiempos_resp), np.array(tiempos_ref)


def simular_IV(t_medio, n_reforzadores=30, tasa_base=1.8, seed=42):
    """
    Intervalo Variable (IV-t).
    El reforzador se vuelve disponible tras un intervalo aleatorio
    con distribución exponencial de media t_medio segundos (correcto para
    programas de IV estándar de laboratorio). Se entrega con la primera
    respuesta después de que el intervalo transcurrió.
    Patrón esperado: pendiente moderada y estable.
    """
    rng = np.random.default_rng(seed)
    tiempos_resp, tiempos_ref = [], []
    t = 0.0
    for _ in range(n_reforzadores):
        t_disponible = t + rng.exponential(t_medio)
        while t < t_disponible:
            t += rng.exponential(1.0 / tasa_base)
            tiempos_resp.append(t)
        tiempos_ref.append(t)   # la última respuesta (t ≥ t_disponible) recoge el reforzador
    return np.array(tiempos_resp), np.array(tiempos_ref)


def simular_IF(t_fijo, n_reforzadores=30, tasa_max=4.0, seed=42):
    """
    Intervalo Fijo (IF-t).
    El reforzador se vuelve disponible exactamente cada t_fijo segundos.
    El modelo produce festoneo (scallop): tasa muy baja durante los primeros
    ~55% del intervalo, luego aceleración sigmoidal hasta la disponibilidad.
    Patrón esperado: pausa + aceleración curvilínea en cada intervalo.
    """
    rng = np.random.default_rng(seed)
    tiempos_resp, tiempos_ref = [], []
    t = 0.0
    for _ in range(n_reforzadores):
        t_inicio     = t
        t_disponible = t_inicio + t_fijo
        while t < t_disponible:
            fraccion = (t - t_inicio) / t_fijo        # 0 → 1 dentro del intervalo
            # Festoneo: tasa mínima en primeros 55%, sigmoide después
            if fraccion < 0.55:
                tasa = tasa_max * 0.04 * (fraccion / 0.55) ** 3
            else:
                tasa = tasa_max / (1.0 + np.exp(-10.0 * (fraccion - 0.78)))
            tasa = max(0.04, tasa)   # piso para evitar esperas infinitas
            t += rng.exponential(1.0 / tasa)
            if t < t_disponible:
                tiempos_resp.append(t)
        # Primera respuesta después de t_disponible recoge el reforzador
        t = t_disponible + rng.exponential(1.0 / tasa_max)
        tiempos_resp.append(t)
        tiempos_ref.append(t)
    return np.array(tiempos_resp), np.array(tiempos_ref)


# ──────────────────────────────────────────────────────────────────────────────
# Funciones de graficación
# ──────────────────────────────────────────────────────────────────────────────

def _graficar_acumulativo(ax, tiempos_resp, tiempos_ref,
                           color, titulo, parametro_str):
    """Dibuja un registro acumulativo con marcas de reforzador."""
    if len(tiempos_resp) == 0:
        ax.set_title(titulo, fontsize=11, fontweight='bold', color=color)
        return

    # Registro acumulativo
    t_plot = np.concatenate([[0.0], tiempos_resp])
    c_plot = np.arange(len(t_plot))
    ax.plot(t_plot / 60.0, c_plot, color=color, linewidth=1.8)

    # Marcas diagonales de reforzador (tick hacia abajo-derecha)
    tick_dx = (t_plot[-1] / 60.0) * 0.012
    tick_dy = c_plot[-1] * 0.030
    for t_ref in tiempos_ref:
        idx = np.searchsorted(tiempos_resp, t_ref)
        y0  = idx + 1
        ax.plot([t_ref / 60.0, t_ref / 60.0 + tick_dx],
                [y0, y0 - tick_dy],
                color=NARANJA, linewidth=1.8)

    # Estadísticas
    dur  = tiempos_resp[-1]
    tr   = len(tiempos_resp) / dur * 60.0
    trf  = len(tiempos_ref)  / dur * 60.0
    txt  = f'Resp/min: {tr:.1f}\nRef/min:  {trf:.2f}\nResp/ref: {tr/trf:.1f}'
    ax.text(0.97, 0.05, txt, transform=ax.transAxes,
            ha='right', va='bottom', fontsize=8.5, color=GRIS,
            family='monospace',
            bbox=dict(facecolor=BLANCO, edgecolor=GRIS_MED,
                      boxstyle='round,pad=0.35', alpha=0.93))

    ax.set_title(f'{titulo}  [{parametro_str}]',
                 fontsize=11, fontweight='bold', color=color, pad=5)
    ax.set_xlabel('Tiempo (min)', fontsize=9, color=GRIS)
    ax.set_ylabel('Respuestas acumuladas', fontsize=9, color=GRIS)
    ax.tick_params(colors=GRIS, labelsize=8)
    ax.set_xlim(0, t_plot[-1] / 60.0)
    ax.set_ylim(0)
    ax.grid(True, alpha=0.18, color=GRIS_MED)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRIS_CLAR)
    ax.set_facecolor(BLANCO)


def _graficar_retroalimentacion(ax_iv, ax_cmp, rv_n, iv_t):
    """
    Dos paneles para la función de retroalimentación — Rachlin (1978): r = d · Rᵐ

    Problema de visualización en un solo panel: con parámetros típicos, la
    asíntota de IV representa ≈ 2-5% del eje Y compartido con RV, haciendo
    la curva IV prácticamente invisible. Solución: dos paneles complementarios.

    Panel izquierdo (ax_iv) — "Curva IV: forma y saturación"
        Muestra únicamente IV en su escala natural (R = 0–300 resp/min).
        Permite ver la forma cóncava completa: crecimiento rápido a tasas bajas,
        desaceleración progresiva, asíntota. Marcadores de 50%, 71% y 90% de
        saturación (fijos en R = 60, 120 y 194 resp/min por la calibración de d).

    Panel derecho (ax_cmp) — "Comparación RV vs IV (rango completo)"
        Ambas curvas, R = 0–600 resp/min. La curva IV se grafica con fill_between
        (banda rellena) para que sea perceptible incluso cuando ocupa solo una
        fracción del eje Y. Comunica la divergencia: RV sigue creciendo sin límite;
        IV se aplana en su techo.

    Parámetros del modelo:
      · RV-n: m = 1   (lineal exacta, d = 1/n)
      · IV-t: m = 0.5 (cóncava, d calibrado → r_max en R = 240 resp/min)
    Los puntos de saturación de IV (50%, 71%, 90%) son invariantes respecto
    a iv_t: R = 60, 120 y 194 resp/min respectivamente.
    """
    # ── Parámetros del modelo ─────────────────────────────────────────────
    r_max_iv = 60.0 / iv_t             # asíntota teórica (ref/min)
    m_iv     = 0.5
    d_iv     = r_max_iv / (240.0 ** m_iv)   # calibrado: r_iv(240) = r_max

    def f_iv(R): return np.minimum(d_iv * R ** m_iv, r_max_iv)
    def f_rv(R): return R / rv_n

    # Puntos de saturación invariantes (por calibración de d):
    # f_iv(R_sat) = p * r_max → d * R^0.5 = p*r_max → R = (p*240^0.5)^2 = p^2*240
    R_50  = 0.25 * 240.0   # 60  resp/min
    R_71  = 0.50 * 240.0   # 120 resp/min
    R_90  = 0.81 * 240.0   # 194 resp/min

    # ── PANEL IZQUIERDO: curva IV en escala propia ────────────────────────
    R_iv = np.linspace(0.5, 300, 500)
    r_iv_plot = f_iv(R_iv)

    ax_iv.plot(R_iv, r_iv_plot, color=COLOR_IV, linewidth=2.4, linestyle='--',
               label=f'IV-{iv_t}s  (m = 0.5)')
    ax_iv.fill_between(R_iv, 0, r_iv_plot, color=COLOR_IV, alpha=0.08)

    # Asíntota
    ax_iv.axhline(r_max_iv, color=COLOR_IV, linewidth=1.0, linestyle=':', alpha=0.70)
    ax_iv.text(295, r_max_iv * 1.04,
               f'r_máx = {r_max_iv:.3f} ref/min  (asíntota = 60 ÷ {iv_t})',
               ha='right', fontsize=8.0, color=COLOR_IV)

    # Marcadores de saturación (líneas verticales + etiquetas)
    for R_sat, pct, va in [(R_50, '50%', 'bottom'), (R_71, '71%', 'bottom'),
                            (R_90, '90%', 'bottom')]:
        if R_sat <= 300:
            r_at = f_iv(R_sat)
            ax_iv.plot([R_sat, R_sat], [0, r_at],
                       color=GRIS_MED, linewidth=0.9, linestyle=':', alpha=0.75)
            ax_iv.plot(R_sat, r_at, 'o', color=COLOR_IV, markersize=5, zorder=5)
            ax_iv.text(R_sat + 4, r_at * 0.35,
                       f'R={R_sat:.0f}\n({pct})',
                       fontsize=7.5, color=GRIS, va='center')

    ax_iv.set_xlim(0, 300)
    ax_iv.set_ylim(0, r_max_iv * 1.25)
    ax_iv.set_title(
        f'IV-{iv_t}s — forma de la curva  (Rachlin m = 0.5)',
        fontsize=10, color=COLOR_IV, fontweight='bold', pad=5)
    ax_iv.set_xlabel('Tasa de respuesta (resp/min)', fontsize=9, color=GRIS)
    ax_iv.set_ylabel('Tasa de reforzamiento (ref/min)', fontsize=9, color=GRIS)
    ax_iv.legend(fontsize=9, framealpha=0.9, edgecolor=GRIS_MED, loc='lower right')
    ax_iv.tick_params(colors=GRIS, labelsize=8)
    ax_iv.grid(True, alpha=0.18, color=GRIS_MED)
    for sp in ax_iv.spines.values():
        sp.set_edgecolor(GRIS_CLAR)
    ax_iv.set_facecolor(BLANCO)

    # ── PANEL DERECHO: comparación RV vs IV, rango completo ──────────────
    R_cmp = np.linspace(0.5, 600, 600)
    r_rv_cmp = f_rv(R_cmp)
    r_iv_cmp = f_iv(R_cmp)

    # RV: línea sólida
    ax_cmp.plot(R_cmp, r_rv_cmp, color=COLOR_RV, linewidth=2.2,
                label=f'RV-{rv_n}  (m = 1,  r = R ÷ {rv_n})')

    # IV: banda rellena + línea encima → visible aunque pequeña en escala relativa
    ax_cmp.fill_between(R_cmp, 0, r_iv_cmp, color=COLOR_IV, alpha=0.22,
                         label=f'IV-{iv_t}s  (m = 0.5,  techo = {r_max_iv:.2f} ref/min)')
    ax_cmp.plot(R_cmp, r_iv_cmp, color=COLOR_IV, linewidth=1.8, linestyle='--')

    # Asíntota IV con flecha de anotación
    ax_cmp.axhline(r_max_iv, color=COLOR_IV, linewidth=1.0, linestyle=':', alpha=0.60)
    ax_cmp.annotate(
        f'asíntota IV = {r_max_iv:.2f} ref/min',
        xy=(580, r_max_iv),
        xytext=(420, r_max_iv + (r_rv_cmp[-1] * 0.12)),
        fontsize=8.0, color=COLOR_IV, ha='center',
        arrowprops=dict(arrowstyle='->', color=COLOR_IV,
                        connectionstyle='arc3,rad=0.2', lw=1.1),
    )

    ax_cmp.set_xlim(0, 600)
    ax_cmp.set_ylim(0)
    ax_cmp.set_title(
        f'RV-{rv_n} vs IV-{iv_t}s — rango completo  (Rachlin 1978: r = d · Rᵐ)',
        fontsize=10, color=GRIS, pad=5)
    ax_cmp.set_xlabel('Tasa de respuesta (resp/min)', fontsize=9, color=GRIS)
    ax_cmp.set_ylabel('Tasa de reforzamiento (ref/min)', fontsize=9, color=GRIS)
    ax_cmp.legend(fontsize=9, framealpha=0.9, edgecolor=GRIS_MED, loc='upper left')
    ax_cmp.tick_params(colors=GRIS, labelsize=8)
    ax_cmp.grid(True, alpha=0.18, color=GRIS_MED)
    for sp in ax_cmp.spines.values():
        sp.set_edgecolor(GRIS_CLAR)
    ax_cmp.set_facecolor(BLANCO)


def actualizar_figura(rv_n, rf_n, iv_t, if_t, n_ref, semilla):
    """Genera la figura completa con cuatro registros + panel retroalimentación."""
    rv_resp, rv_ref = simular_RV(rv_n, n_ref, seed=semilla)
    rf_resp, rf_ref = simular_RF(rf_n, n_ref, seed=semilla)
    iv_resp, iv_ref = simular_IV(iv_t, n_ref, seed=semilla)
    if_resp, if_ref = simular_IF(if_t, n_ref, seed=semilla)

    fig = plt.figure(figsize=(14, 9.5), facecolor=BLANCO)
    fig.patch.set_facecolor(BLANCO)

    # Borde superior azul (estilo del libro)
    fig.add_artist(plt.Line2D([0.03, 0.97], [0.988, 0.988],
                               transform=fig.transFigure,
                               color=AZUL, linewidth=3.0))

    gs = GridSpec(3, 2, figure=fig,
                  hspace=0.54, wspace=0.32,
                  top=0.93, bottom=0.07, left=0.07, right=0.97)

    _graficar_acumulativo(
        fig.add_subplot(gs[0, 0]), rv_resp, rv_ref,
        COLOR_RV, 'Razón Variable (RV)', f'n = {rv_n}')
    _graficar_acumulativo(
        fig.add_subplot(gs[0, 1]), rf_resp, rf_ref,
        COLOR_RF, 'Razón Fija (RF)', f'n = {rf_n}')
    _graficar_acumulativo(
        fig.add_subplot(gs[1, 0]), iv_resp, iv_ref,
        COLOR_IV, 'Intervalo Variable (IV)', f't = {iv_t} s')
    _graficar_acumulativo(
        fig.add_subplot(gs[1, 1]), if_resp, if_ref,
        COLOR_IF, 'Intervalo Fijo (IF)', f't = {if_t} s')
    _graficar_retroalimentacion(
        fig.add_subplot(gs[2, 0]),   # panel IV: forma y saturación
        fig.add_subplot(gs[2, 1]),   # panel comparación: RV vs IV completo
        rv_n, iv_t)

    # Leyenda en el espacio en blanco entre la fila de registros (IV/IF)
    # y la fila de retroalimentación. y=0.338 = centro exacto del gap
    # calculado desde los parámetros del GridSpec (top=0.93, bottom=0.07,
    # hspace=0.54, 3 filas). No solapa ningún panel, título ni encabezado.
    fig.legend(
        handles=[mpatches.Patch(color=AZUL,   label='Respuesta acumulada'),
                 mpatches.Patch(color=NARANJA, label='Entrega de reforzador')],
        loc='center', ncol=2, fontsize=9,
        framealpha=0.92, edgecolor=GRIS_MED,
        bbox_to_anchor=(0.5, 0.338))

    fig.text(0.5, 0.972, 'Simulador 13.1 · Programas de Refuerzo',
             ha='center', va='top', fontsize=13,
             fontweight='bold', color=AZUL)
    plt.show()


# ──────────────────────────────────────────────────────────────────────────────
# Interfaz interactiva
# Ajusta los sliders y presiona ▶ Simular.
# El render inicial ocurre automáticamente con los valores predeterminados.
# ──────────────────────────────────────────────────────────────────────────────

def crear_interfaz():
    kw = dict(style={'description_width': '200px'},
              layout=widgets.Layout(width='460px'))

    # ── Widgets HTML de estructura (estilo referencia cap. 11) ────────────
    w_header        = widgets.HTML(value=_html_header_131())
    w_sec_razon     = widgets.HTML(value=_html_sec_131('Programas de Razón'))
    w_sec_intervalo = widgets.HTML(value=_html_sec_131('Programas de Intervalo'))
    w_sec_general   = widgets.HTML(value=_html_sec_131('Configuración General'))

    # ── Sliders ───────────────────────────────────────────────────────────
    sl_rv_n = widgets.IntSlider(
        value=15, min=5,  max=60,  step=5,
        description='RV — requisito medio (n):', **kw)
    sl_rf_n = widgets.IntSlider(
        value=15, min=5,  max=60,  step=5,
        description='RF — requisito fijo (n):', **kw)
    sl_iv_t = widgets.IntSlider(
        value=60, min=15, max=300, step=15,
        description='IV — intervalo medio (s):', **kw)
    sl_if_t = widgets.IntSlider(
        value=60, min=15, max=300, step=15,
        description='IF — intervalo fijo (s):', **kw)
    sl_nref = widgets.IntSlider(
        value=25, min=10, max=50, step=5,
        description='Reforzadores por sesión:', **kw)
    sl_seed = widgets.IntSlider(
        value=42, min=1, max=200, step=1,
        description='Semilla aleatoria:', **kw)

    boton = widgets.Button(
        description='▶  Simular',
        layout=widgets.Layout(width='175px', height='40px'),
        style={'button_color': AZUL, 'font_weight': 'bold'})

    salida = widgets.Output()

    # ── Columnas de controles ─────────────────────────────────────────────
    col_izq = widgets.VBox(
        [w_sec_razon, sl_rv_n, sl_rf_n,
         widgets.HTML('<br>'),
         w_sec_general, sl_nref, sl_seed, boton],
        layout=widgets.Layout(padding='4px 16px'))
    col_der = widgets.VBox(
        [w_sec_intervalo, sl_iv_t, sl_if_t],
        layout=widgets.Layout(padding='4px 16px'))

    # ── Cuerpo del panel (con fondo y borde, estilo cap. 11) ──────────────
    cuerpo = widgets.VBox(
        [widgets.HBox([col_izq, col_der])],
        layout=widgets.Layout(
            padding='10px 16px 14px 16px',
            background_color=PANEL_BG,
            border=f'1px solid {PANEL_BRD}',
            border_radius='0 0 6px 6px',
        ))

    ui = widgets.VBox([w_header, cuerpo])

    # ── Callback ──────────────────────────────────────────────────────────
    def _run(_):
        with salida:
            clear_output(wait=True)
            actualizar_figura(
                rv_n    = sl_rv_n.value,
                rf_n    = sl_rf_n.value,
                iv_t    = sl_iv_t.value,
                if_t    = sl_if_t.value,
                n_ref   = sl_nref.value,
                semilla = sl_seed.value)

    boton.on_click(_run)
    display(ui, salida)
    _run(None)   # Render inicial automático


crear_interfaz()

Output()

---

## Ejercicios

Antes de cambiar los parámetros, **registra tu predicción**. Luego verifica con la simulación.

---

### Ejercicio 1 · Básico — Efecto del tamaño del requisito

Configura **RF-10** y **RV-10**. Luego aumenta a **RF-40** y **RV-40**.

**a)** ¿Cómo cambia la *pendiente* del registro cuando aumenta el requisito? ¿Qué significa en términos de tasa de respuesta?

**b)** ¿Qué le ocurre a la *pausa post-reforzador* en RF cuando el requisito aumenta? Exprésalo en segundos usando las estadísticas del panel.

**c)** ¿Observas pausas post-reforzador bajo RV? ¿Por qué sí o no?

> *Predicción antes de simular:* "Cuando el requisito sube de 10 a 40, la pausa en RF aumentará / disminuirá / permanecerá igual porque ___________."

---

### Ejercicio 2 · Intermedio — Razón vs. intervalo con reforzamiento igualado

El experimento de cajas acopladas (Catania, 1971) demostró que la diferencia RV vs. IV no se debe a diferencias en la tasa de reforzamiento.

1. Configura **RV-20** e **IV-60s**. Anota *Resp/min* y *Ref/min* para cada programa.
2. Ajusta el intervalo del IV hasta que su *Ref/min* se aproxime al *Ref/min* del RV-20.
3. Con tasas de reforzamiento similares, ¿la diferencia en *Resp/min* persiste?

**a)** ¿Cuál programa sigue produciendo mayor tasa de respuesta a pesar del igualamiento?

**b)** En el panel inferior, ¿qué diferencia entre la curva de RV y la de IV explica este resultado?

---

### Ejercicio 3 · Intermedio — El festoneo en IF

Configura IF con intervalos de **30, 60, 120 y 240 segundos** (uno a la vez).

**a)** ¿Cómo cambia la *forma* del registro (el festoneo) cuando el intervalo aumenta? ¿Las pausas son proporcionalmente iguales al intervalo?

**b)** Compara la pendiente de **IF-60s** con la de **IV-60s**. ¿Cuál es más estable? ¿Por qué?

**c)** El panel de estadísticas muestra *Resp/ref*. ¿Qué programa "desperdicia" más respuestas sin obtener reforzador, IF o IV con el mismo valor de *t*?

---

### Ejercicio 4 · Avanzado — Función de retroalimentación

**a)** Con **RV-10** e **IV-30s**: ¿cuántas respuestas adicionales por minuto necesita el organismo bajo RV para *duplicar* su tasa de reforzamiento? ¿Y bajo IV?

**b)** Con **RV-40** e **IV-120s**: ¿en qué región de la curva de IV opera un organismo con tasa baja (< 30 resp/min)? ¿Y con tasa alta (> 150 resp/min)?

**c)** A una tasa de respuesta fija de 60 resp/min, ¿qué programa da más reforzadores? Verifica calculando con las fórmulas del capítulo:
- RV: $r = R/n$
- IV (aproximación): $r = \dfrac{(1/t) \cdot R}{R + (1/t)}$

---

### Ejercicio 5 · Reflexión — Programas y vida cotidiana

Sin usar el simulador, clasifica las siguientes situaciones. Justifica cada clasificación en términos del **criterio** (respuestas vs. tiempo) y la **fijeza** (fijo vs. variable). Luego predice: ¿pausas post-reforzador? ¿Tasa alta o moderada? ¿Festoneo?

| Situación | Programa | ¿Pausas? | ¿Tasa? |
|-----------|----------|----------|--------|
| Revisar el correo esperando un mensaje importante | | | |
| Vendedor con comisión por cada venta cerrada | | | |
| Estudiar para exámenes con fechas publicadas | | | |
| Pescar con caña en un lago | | | |
| Series de pesas: 12 repeticiones, descanso, siguiente serie | | | |
| Intentar hacer reír a alguien con chistes | | | |


In [ ]:
#@title **Simulador 13.2** — Reforzamiento diferencial de tiempos entre respuestas


# ——— SET UP ———————————————————————————————————————————————————————————————————
# En Google Colab, ipywidgets suele estar disponible.
try:
    import ipywidgets
    print("✓ ipywidgets", ipywidgets.__version__)
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "ipywidgets", "--quiet"])
    print("ipywidgets instalado — reinicia el runtime y vuelve a ejecutar.")

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
import ipywidgets as widgets
from IPython.display import display, clear_output

matplotlib.rcParams.update({
    'font.family':      'serif',
    'font.serif':       ['Georgia', 'Palatino Linotype', 'DejaVu Serif'],
    'axes.spines.top':  False,
    'axes.spines.right': False,
})

# ── Paleta del libro ──────────────────────────────────────────────────────
AZUL     = '#2C5282'
NARANJA  = '#C05621'
VERDE    = '#276749'
GRIS     = '#718096'
GRIS_MED = '#A0AEC0'
GRIS_CL  = '#EDF2F7'
BLANCO   = '#FFFFFF'
COLOR_RV = AZUL
COLOR_IV = NARANJA

print("✓ Importaciones y paleta configuradas.")


# ─────────────────────────────────────────────────────────────────────────
# FUNCIONES DE SIMULACIÓN
# ─────────────────────────────────────────────────────────────────────────

def generar_irts_rafaga(n_irts, irt_corto, irt_largo, tam_rafaga, seed):
    """
    Genera n_irts TERs con estructura explícita de ráfaga:
      · Dentro de ráfaga: TER ~ Exp(irt_corto)
      · Primera respuesta de nueva ráfaga: TER ~ Exp(irt_largo)
      · Tamaño de ráfaga: Geométrica(1/tam_rafaga)

    Devuelve:
        irts     : array de n_irts TERs
        tiempos  : array de n_irts tiempos acumulados (tiempo de cada respuesta)
        es_corto : bool array, True si el TER es within-burst (corto)
    """
    rng = np.random.default_rng(seed)
    irts     = np.empty(n_irts)
    es_corto = np.zeros(n_irts, dtype=bool)

    i = 0
    t = 0.0
    while i < n_irts:
        # ── Nueva ráfaga: el primer TER es LARGO (pausa entre ráfagas) ──
        if i > 0:                          # no hay TER antes de la primera respuesta
            irt = rng.exponential(irt_largo)
            irts[i]     = irt
            es_corto[i] = False
            t += irt
            i += 1
            if i >= n_irts:
                break

        # ── Tamaño de esta ráfaga ─────────────────────────────────────
        tam = max(1, int(rng.geometric(1.0 / tam_rafaga)))

        # ── Respuestas dentro de la ráfaga: TER CORTO ─────────────────
        for _ in range(tam - 1):
            if i >= n_irts:
                break
            irt = rng.exponential(irt_corto)
            irts[i]     = irt
            es_corto[i] = True
            t += irt
            i += 1

    tiempos = np.cumsum(irts)
    return irts, tiempos, es_corto


def encontrar_reforzados_rv(irts, rv_n, seed):
    """
    RV-n: reforzar cada ~rv_n respuestas (requisito ~ Geométrica con media rv_n).
    Devuelve los TERs que precedieron a una respuesta reforzada.
    """
    rng = np.random.default_rng(seed)
    ref = []
    req   = max(1, int(rng.geometric(1.0 / rv_n)))
    count = 0
    for k in range(len(irts)):
        count += 1
        if count >= req:
            ref.append(irts[k])
            count = 0
            req = max(1, int(rng.geometric(1.0 / rv_n)))
    return np.array(ref) if ref else np.array([1e-3])


def encontrar_reforzados_iv(irts, tiempos, iv_t, seed):
    """
    IV-t: reforzar la primera respuesta tras un intervalo ~Exp(iv_t).
    Devuelve los TERs que precedieron a una respuesta reforzada.
    """
    rng = np.random.default_rng(seed)
    ref    = []
    t_prox = rng.exponential(iv_t)
    for k in range(len(irts)):
        if tiempos[k] >= t_prox:
            ref.append(irts[k])
            t_prox = tiempos[k] + rng.exponential(iv_t)
    return np.array(ref) if ref else np.array([1e-3])


def prob_relativa_reforzamiento(irts_all, irts_ref, bins):
    """
    Probabilidad relativa de reforzamiento como función del TER:
        P(reforzado | TER ∈ bin) / P(reforzado)
    Valores > 1: sobre-representación; < 1: sub-representación.
    Una curva plana en 1.0 indicaría selección proporcional a la base.
    """
    n_all, _ = np.histogram(irts_all, bins=bins)
    n_ref, _ = np.histogram(irts_ref, bins=bins)
    tasa_base = len(irts_ref) / len(irts_all)
    with np.errstate(divide='ignore', invalid='ignore'):
        prob = np.where(n_all > 5,
                        (n_ref / n_all) / tasa_base,
                        np.nan)
    centros = np.sqrt(bins[:-1] * bins[1:])   # media geométrica del bin
    return centros, prob


def simular_evolucion(irt_corto, irt_largo, tam_rafaga,
                       rv_n, iv_t, alpha, n_epochs,
                       n_por_epoch=1200, seed_base=42):
    """
    Simula n_epochs bloques de entrenamiento.
    Parámetro adaptativo: tam_rafaga (tamaño medio de ráfaga).
      · RV refuerza TERs cortos → tam_rafaga crece → más respuestas por ráfaga
      · IV refuerza TERs largos → tam_rafaga decrece → ráfagas más cortas
    (La pausa entre ráfagas irt_largo se mantiene fija para aislar el efecto.)

    Devuelve:
        historial con p_corto (proporción de TERs cortos) por bloque,
        y snapshots de la distribución de TERs en bloques seleccionados.
    """
    tam_rv = float(tam_rafaga)
    tam_iv = float(tam_rafaga)
    threshold = np.sqrt(irt_corto * irt_largo)

    def p_corto_de_tam(tam):
        "Proporción teórica de TERs cortos dado el tamaño medio de ráfaga."
        return (tam - 1) / tam if tam > 1 else 0.0

    hist = {
        'p_rv': [p_corto_de_tam(tam_rv)],
        'p_iv': [p_corto_de_tam(tam_iv)],
        'snaps_rv': {},
        'snaps_iv': {},
    }
    snap_set = {0, n_epochs // 3, 2 * n_epochs // 3, n_epochs - 1}

    for ep in range(n_epochs):
        s = seed_base + ep

        irts_rv, t_rv, _ = generar_irts_rafaga(n_por_epoch, irt_corto, irt_largo,
                                                tam_rv, s)
        irts_iv, t_iv, _ = generar_irts_rafaga(n_por_epoch, irt_corto, irt_largo,
                                                tam_iv, s + 5000)

        ref_rv = encontrar_reforzados_rv(irts_rv, rv_n, s + 10000)
        ref_iv = encontrar_reforzados_iv(irts_iv, t_iv, iv_t, s + 20000)

        if ep in snap_set:
            hist['snaps_rv'][ep] = irts_rv.copy()
            hist['snaps_iv'][ep] = irts_iv.copy()

        # Fracción de TERs reforzados que son CORTOS
        psr = float(np.mean(ref_rv < threshold))
        psi = float(np.mean(ref_iv < threshold))

        # El tamaño de ráfaga se ajusta proporcionalmente:
        # alta psr (refuerza cortos) → ráfagas más largas (más TERs cortos)
        # baja psi (refuerza largos) → ráfagas más cortas (más TERs largos)
        tam_rv = np.clip((1 - alpha) * tam_rv + alpha * (1 + psr * (tam_rafaga * 2 - 1)),
                         1.5, tam_rafaga * 5)
        tam_iv = np.clip((1 - alpha) * tam_iv + alpha * (1 + psi  * (tam_rafaga * 2 - 1)),
                         1.5, tam_rafaga * 5)

        hist['p_rv'].append(p_corto_de_tam(tam_rv))
        hist['p_iv'].append(p_corto_de_tam(tam_iv))

    hist['threshold'] = threshold
    return hist

print("✓ Funciones de simulación definidas.")


# ─────────────────────────────────────────────────────────────────────────
# FUNCIONES DE GRAFICACIÓN
# ─────────────────────────────────────────────────────────────────────────

def _estilo_ax(ax):
    ax.tick_params(colors=GRIS, labelsize=8)
    ax.grid(True, alpha=0.14, color=GRIS_MED)
    ax.set_facecolor(BLANCO)
    for sp in ax.spines.values():
        sp.set_edgecolor(GRIS_CL)


def _bins(irt_corto, irt_largo, n=50):
    lo = max(irt_corto * 0.05, 0.01)
    hi = irt_largo * 30
    return np.geomspace(lo, hi, n + 1)


def _panel_histograma(ax, irts_all, irts_ref, color, titulo,
                       threshold, irt_corto, irt_largo, bins):
    """
    Panel superior: todos los TERs (gris) + TERs reforzados (color).
    Escala log en el eje X para visualizar la distribución bimodal.
    """
    ax.hist(np.clip(irts_all, bins[0], bins[-1]),
            bins=bins, density=True, color=GRIS, alpha=0.40,
            label='Todos los TERs')
    ax.hist(np.clip(irts_ref, bins[0], bins[-1]),
            bins=bins, density=True, color=color, alpha=0.80,
            label='TERs reforzados')

    # Líneas de referencia
    ax.axvline(threshold,  color='#2d2d2d', linewidth=1.3, linestyle='--', alpha=0.6,
               label=f'umbral ({threshold:.2f} s)')
    ax.axvline(irt_corto,  color=color,      linewidth=0.9, linestyle=':', alpha=0.55)
    ax.axvline(irt_largo,  color=GRIS,       linewidth=0.9, linestyle=':', alpha=0.55)

    # Etiquetas de los modos
    ymax = ax.get_ylim()[1]
    ax.text(irt_corto * 1.15, ymax * 0.92,
            f'TER corto\n(μ={irt_corto:.2f}s)',
            fontsize=7.5, color=color, alpha=0.8, ha='left')
    ax.text(irt_largo * 1.15, ymax * 0.92,
            f'TER largo\n(μ={irt_largo:.1f}s)',
            fontsize=7.5, color=GRIS, alpha=0.8, ha='left')

    # Estadísticas
    pct_c = 100.0 * np.mean(irts_ref < threshold)
    ax.text(0.97, 0.06,
            f'Reforzados cortos: {pct_c:.0f}%\nReforzados largos: {100-pct_c:.0f}%',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=9, color=color, fontweight='bold',
            bbox=dict(facecolor=BLANCO, edgecolor=color,
                      boxstyle='round,pad=0.35', alpha=0.93))

    ax.set_xscale('log')
    ax.set_xlabel('TER (s, escala log)', fontsize=9, color=GRIS)
    ax.set_ylabel('Densidad relativa', fontsize=9, color=GRIS)
    ax.set_title(titulo, fontsize=11, fontweight='bold', color=color, pad=5)
    ax.legend(fontsize=7.5, framealpha=0.9, edgecolor=GRIS_MED)
    _estilo_ax(ax)


def _panel_prob_cond(ax, centros_rv, prob_rv, centros_iv, prob_iv, threshold):
    """
    Panel medio: probabilidad RELATIVA de reforzamiento en función del TER.
    Valor 1 = tasa de reforzamiento proporcional a la frecuencia base.
    RV ≈ línea plana; IV sube para TERs largos.
    """
    # Eliminar NaNs para la gráfica
    m_rv = ~np.isnan(prob_rv)
    m_iv = ~np.isnan(prob_iv)

    ax.plot(centros_rv[m_rv], prob_rv[m_rv],
            color=COLOR_RV, linewidth=2.2, label='RV', zorder=4)
    ax.plot(centros_iv[m_iv], prob_iv[m_iv],
            color=COLOR_IV, linewidth=2.2, linestyle='--', label='IV', zorder=4)

    ax.axhline(1.0, color='#2d2d2d', linewidth=1.1, linestyle=':', alpha=0.55,
               label='nivel de azar')
    ax.axvline(threshold, color='#2d2d2d', linewidth=1.1, linestyle='--', alpha=0.45)

    ax.fill_between(centros_rv[m_rv], 1, prob_rv[m_rv],
                    where=prob_rv[m_rv] > 1,
                    color=COLOR_RV, alpha=0.12)
    ax.fill_between(centros_iv[m_iv], 1, prob_iv[m_iv],
                    where=prob_iv[m_iv] > 1,
                    color=COLOR_IV, alpha=0.12)

    ax.set_xscale('log')
    ax.set_xlabel('TER (s, escala log)', fontsize=9, color=GRIS)
    ax.set_ylabel('P(reforzado|TER) / P(reforzado)', fontsize=9, color=GRIS)
    ax.set_title('Probabilidad relativa de reforzamiento por TER',
                 fontsize=10, color=GRIS, fontweight='bold', pad=4)
    ax.legend(fontsize=9, framealpha=0.9, edgecolor=GRIS_MED)
    ax.set_ylim(0)
    _estilo_ax(ax)

    # Anotaciones explicativas
    ax.text(0.02, 0.93,
            'RV ≈ plano (selección proporcional a la frecuencia base)',
            transform=ax.transAxes, fontsize=8, color=COLOR_RV, va='top')
    ax.text(0.02, 0.83,
            'IV ↑ para TERs largos (selección sesgada)',
            transform=ax.transAxes, fontsize=8, color=COLOR_IV, va='top')


def _panel_evolucion(ax, hist, p0, rv_n, iv_t):
    """
    Panel inferior izquierdo: proporción de TERs cortos por bloque de entrenamiento.
    """
    x = range(len(hist['p_rv']))
    ax.plot(x, hist['p_rv'], color=COLOR_RV, linewidth=2.3,
            label=f'RV-{rv_n}', solid_capstyle='round')
    ax.plot(x, hist['p_iv'], color=COLOR_IV, linewidth=2.3,
            linestyle='--', label=f'IV-{iv_t}s', solid_capstyle='round')
    ax.axhline(p0, color=GRIS, linewidth=1.1, linestyle=':', alpha=0.65,
               label=f'línea base (p₀={p0:.2f})')

    # Anotar valores finales
    xf = len(x) - 1
    p_rv_f = hist['p_rv'][-1]
    p_iv_f = hist['p_iv'][-1]
    ax.annotate(f'{p_rv_f:.2f}',
                xy=(xf, p_rv_f), xytext=(xf - max(1, xf//5), p_rv_f + 0.06),
                fontsize=9, color=COLOR_RV, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=COLOR_RV, lw=1.1))
    ax.annotate(f'{p_iv_f:.2f}',
                xy=(xf, p_iv_f), xytext=(xf - max(1, xf//5), p_iv_f - 0.08),
                fontsize=9, color=COLOR_IV, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=COLOR_IV, lw=1.1))

    ax.set_xlabel('Bloque de entrenamiento', fontsize=9, color=GRIS)
    ax.set_ylabel('p(TER corto) en el comportamiento', fontsize=9, color=GRIS)
    ax.set_title('Evolución de la distribución de TERs',
                 fontsize=10, color=GRIS, fontweight='bold', pad=4)
    ax.legend(fontsize=9, framealpha=0.9, edgecolor=GRIS_MED)
    ax.set_ylim(0, 1)
    ax.set_xlim(0, len(x) - 1)
    _estilo_ax(ax)


def _panel_desplazamiento(ax, snaps, color, titulo, bins, threshold):
    """
    Panel inferior derecho: distribución de TERs en 4 bloques seleccionados.
    El desplazamiento de la distribución muestra el efecto del entrenamiento.
    """
    epochs  = sorted(snaps.keys())
    alphas  = np.linspace(0.28, 0.92, len(epochs))
    n_total = max(ep for ep in epochs) if epochs else 1

    for i, ep in enumerate(epochs):
        irts = snaps[ep]
        label = (f'Bloque {ep+1}' if ep == epochs[0] else
                 f'Bloque {ep+1}'  if ep == epochs[-1] else
                 f'Bloque {ep+1}')
        ax.hist(np.clip(irts, bins[0], bins[-1]),
                bins=bins, density=True,
                color=color, alpha=float(alphas[i]),
                label=label, histtype='stepfilled')

    ax.axvline(threshold, color='#2d2d2d', linewidth=1.1,
               linestyle='--', alpha=0.45)
    ax.set_xscale('log')
    ax.set_xlabel('TER (s, escala log)', fontsize=9, color=GRIS)
    ax.set_ylabel('Densidad relativa', fontsize=9, color=GRIS)
    ax.set_title(titulo, fontsize=10, color=color, fontweight='bold', pad=4)
    ax.legend(fontsize=7.5, framealpha=0.9, edgecolor=GRIS_MED)
    _estilo_ax(ax)


def hacer_figura(irt_corto, irt_largo, tam_rafaga,
                  rv_n, iv_t, alpha, n_epochs, seed):
    """
    Genera la figura completa con 4 paneles en 3 filas:
      Fila 0 (L|R): Histogramas todo+reforzado para RV e IV
      Fila 1 (completo): Probabilidad relativa de reforzamiento
      Fila 2 (L|R): Evolución p(TER corto) | Distribución en 4 épocas
    """
    # ── Simulación estática (mismo stream para RV e IV) ───────────────────
    N = 10_000
    irts_all, tiempos, _ = generar_irts_rafaga(N, irt_corto, irt_largo, tam_rafaga, seed)
    ref_rv = encontrar_reforzados_rv(irts_all, rv_n,   seed + 1)
    ref_iv = encontrar_reforzados_iv(irts_all, tiempos, iv_t, seed + 2)

    threshold = np.sqrt(irt_corto * irt_largo)
    bins      = _bins(irt_corto, irt_largo)

    # Probabilidades condicionales
    centros_rv, prob_rv = prob_relativa_reforzamiento(irts_all, ref_rv, bins)
    centros_iv, prob_iv = prob_relativa_reforzamiento(irts_all, ref_iv, bins)

    # ── Simulación de evolución ───────────────────────────────────────────
    evol = simular_evolucion(irt_corto, irt_largo, tam_rafaga,
                              rv_n, iv_t, alpha, n_epochs,
                              seed_base=seed + 100)
    p0_real = (tam_rafaga - 1) / tam_rafaga if tam_rafaga > 1 else 0.0

    # ── Figura ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(14, 13), facecolor=BLANCO)
    fig.patch.set_facecolor(BLANCO)
    fig.add_artist(plt.Line2D([0.03, 0.97], [0.990, 0.990],
                               transform=fig.transFigure,
                               color=AZUL, linewidth=3.0))

    gs = GridSpec(3, 2, figure=fig,
                  height_ratios=[1.7, 1.0, 1.5],
                  hspace=0.55, wspace=0.30,
                  top=0.94, bottom=0.06, left=0.08, right=0.97)

    ax_rv   = fig.add_subplot(gs[0, 0])
    ax_iv   = fig.add_subplot(gs[0, 1])
    ax_prob = fig.add_subplot(gs[1, :])
    ax_evol = fig.add_subplot(gs[2, 0])
    ax_desp = fig.add_subplot(gs[2, 1])

    # ── Fila 0: Histogramas ───────────────────────────────────────────────
    _panel_histograma(
        ax_rv, irts_all, ref_rv, COLOR_RV,
        f'Razón Variable (RV-{rv_n})',
        threshold, irt_corto, irt_largo, bins)

    _panel_histograma(
        ax_iv, irts_all, ref_iv, COLOR_IV,
        f'Intervalo Variable (IV-{iv_t}s)',
        threshold, irt_corto, irt_largo, bins)

    # ── Fila 1: Probabilidad condicional ──────────────────────────────────
    _panel_prob_cond(ax_prob, centros_rv, prob_rv, centros_iv, prob_iv, threshold)

    # ── Fila 2: Evolución y desplazamiento ────────────────────────────────
    _panel_evolucion(ax_evol, evol, p0_real, rv_n, iv_t)

    # Usar snaps de RV para panel de desplazamiento (mostrar RV y IV superpuestos)
    # Panel de desplazamiento: mostrar ambos programas en el mismo panel
    epochs_rv  = sorted(evol['snaps_rv'].keys())
    epochs_iv  = sorted(evol['snaps_iv'].keys())
    alphas_seq = np.linspace(0.28, 0.92, len(epochs_rv))

    ax_desp.set_xscale('log')
    for i, (ep_rv, ep_iv) in enumerate(zip(epochs_rv, epochs_iv)):
        irts_rv_s = evol['snaps_rv'][ep_rv]
        irts_iv_s = evol['snaps_iv'][ep_iv]
        a = float(alphas_seq[i])
        label_rv = f'RV — bloque {ep_rv+1}' if i in (0, len(epochs_rv)-1) else '_nolegend_'
        label_iv = f'IV — bloque {ep_iv+1}' if i in (0, len(epochs_iv)-1) else '_nolegend_'
        ax_desp.hist(np.clip(irts_rv_s, bins[0], bins[-1]),
                     bins=bins, density=True, color=COLOR_RV,
                     alpha=a, histtype='step', linewidth=1.5, label=label_rv)
        ax_desp.hist(np.clip(irts_iv_s, bins[0], bins[-1]),
                     bins=bins, density=True, color=COLOR_IV,
                     alpha=a, histtype='step', linewidth=1.5, linestyle='--',
                     label=label_iv)

    ax_desp.axvline(threshold, color='#2d2d2d', linewidth=1.1,
                    linestyle='--', alpha=0.45)
    ax_desp.set_xlabel('TER (s, escala log)', fontsize=9, color=GRIS)
    ax_desp.set_ylabel('Densidad relativa', fontsize=9, color=GRIS)
    ax_desp.set_title('Desplazamiento de la distribución de TERs',
                      fontsize=10, color=GRIS, fontweight='bold', pad=4)
    ax_desp.legend(fontsize=7.5, framealpha=0.9, edgecolor=GRIS_MED)
    _estilo_ax(ax_desp)

    # ── Leyenda global ────────────────────────────────────────────────────
    fig.legend(
        handles=[
            mpatches.Patch(color=COLOR_RV, label='Razón Variable (RV)'),
            mpatches.Patch(color=COLOR_IV, label='Intervalo Variable (IV)'),
            mpatches.Patch(color=GRIS,     label='Distribución base (todos los TERs)'),
        ],
        loc='center', ncol=3, fontsize=9,
        framealpha=0.9, edgecolor=GRIS_MED,
        bbox_to_anchor=(0.5, 0.625))

    fig.text(0.5, 0.973,
             'Simulador 13.2 · Reforzamiento Diferencial de Tiempos Entre Respuestas',
             ha='center', va='top', fontsize=13, fontweight='bold', color=AZUL)

    plt.show()

print("✓ Funciones de graficación definidas.")

# ─────────────────────────────────────────────────────────────────────────
# INTERFAZ INTERACTIVA
# ─────────────────────────────────────────────────────────────────────────

def crear_interfaz():
    kw = dict(style={'description_width': '200px'},
              layout=widgets.Layout(width='470px'))

    encabezado = widgets.HTML(
        '<div style="background:#2C5282;color:white;padding:10px 18px;' +
        'border-radius:6px;font-family:Georgia,serif;font-size:14px;' +
        'font-weight:bold;margin-bottom:10px">' +
        '🔬 Simulador 13.2 &nbsp;·&nbsp; ' +
        'Reforzamiento Diferencial de TERs<br>' +
        '<span style="font-size:11px;font-weight:normal">' +
        'Ajusta los parámetros y presiona <b>▶ Simular</b>. ' +
        'Registra tu predicción <em>antes</em> de correr la simulación.' +
        '</span></div>'
    )

    # ── Comportamiento base ───────────────────────────────────────────────
    sl_irt_c  = widgets.FloatSlider(
        value=0.3, min=0.05, max=1.5, step=0.05,
        description='TER dentro de ráfaga (s):',
        readout_format='.2f', **kw)
    sl_irt_l  = widgets.FloatSlider(
        value=8.0, min=2.0, max=40.0, step=1.0,
        description='Pausa entre ráfagas (s):',
        readout_format='.1f', **kw)
    sl_tam    = widgets.IntSlider(
        value=5, min=2, max=15, step=1,
        description='Tamaño medio de ráfaga:', **kw)

    # ── Programas ─────────────────────────────────────────────────────────
    sl_rv_n   = widgets.IntSlider(
        value=15, min=5, max=60, step=5,
        description='RV — requisito medio (n):', **kw)
    sl_iv_t   = widgets.IntSlider(
        value=60, min=10, max=240, step=10,
        description='IV — intervalo medio (s):', **kw)

    # ── Evolución ─────────────────────────────────────────────────────────
    sl_alpha  = widgets.FloatSlider(
        value=0.15, min=0.03, max=0.50, step=0.02,
        description='Tasa de aprendizaje α:',
        readout_format='.2f', **kw)
    sl_epochs = widgets.IntSlider(
        value=20, min=5, max=60, step=5,
        description='Bloques de entrenamiento:', **kw)
    sl_seed   = widgets.IntSlider(
        value=42, min=1, max=200, step=1,
        description='Semilla aleatoria:', **kw)

    boton = widgets.Button(
        description='▶  Simular',
        layout=widgets.Layout(width='175px', height='40px'),
        style={'button_color': '#2C5282', 'font_weight': 'bold'})

    salida = widgets.Output()

    # ── Etiqueta con umbral calculado ─────────────────────────────────────
    umbral_label = widgets.HTML(
        f'<span style="color:#718096;font-family:Georgia;font-size:11px">' +
        f'Umbral TER corto/largo (√(TERc·TERl)): ' +
        f'<b>{(sl_irt_c.value * sl_irt_l.value)**0.5:.2f} s</b></span>'
    )

    def _actualizar_umbral(*_):
        v = (sl_irt_c.value * sl_irt_l.value) ** 0.5
        umbral_label.value = (
            '<span style="color:#718096;font-family:Georgia;font-size:11px">' +
            f'Umbral TER corto/largo (√(TERc·TERl)): <b>{v:.2f} s</b></span>')

    sl_irt_c.observe(_actualizar_umbral, names='value')
    sl_irt_l.observe(_actualizar_umbral, names='value')

    # ── Layout ────────────────────────────────────────────────────────────
    sep = lambda txt, color: widgets.HTML(
        f'<b style="color:{color};font-family:Georgia;font-size:12px">{txt}</b>')

    col_izq = widgets.VBox([
        sep('Estructura de ráfaga', AZUL),
        sl_irt_c, sl_irt_l, sl_tam, umbral_label,
        widgets.HTML('<br>'),
        sep('Programas de refuerzo', GRIS),
        sl_rv_n, sl_iv_t,
    ], layout=widgets.Layout(padding='8px 16px'))

    col_der = widgets.VBox([
        sep('Parámetros de evolución', GRIS),
        sl_alpha, sl_epochs, sl_seed,
        widgets.HTML('<br>'),
        boton,
    ], layout=widgets.Layout(padding='8px 16px'))

    controles = widgets.HBox([col_izq, col_der])

    def _run(_):
        with salida:
            clear_output(wait=True)
            hacer_figura(
                irt_corto  = sl_irt_c.value,
                irt_largo  = sl_irt_l.value,
                tam_rafaga = sl_tam.value,
                rv_n       = sl_rv_n.value,
                iv_t       = sl_iv_t.value,
                alpha      = sl_alpha.value,
                n_epochs   = sl_epochs.value,
                seed       = sl_seed.value,
            )

    boton.on_click(_run)
    display(encabezado, controles, salida)
    _run(None)

crear_interfaz()


✓ ipywidgets 7.7.1
✓ Importaciones y paleta configuradas.
✓ Funciones de simulación definidas.
✓ Funciones de graficación definidas.


HTML(value='<div style="background:#2C5282;color:white;padding:10px 18px;border-radius:6px;font-family:Georgia…

Output()

---

## Ejercicios

Registra tu predicción **antes** de cambiar los parámetros. Luego verifica.

---

### Ejercicio 1 · Básico — La asimetría fundamental

Configura los valores predeterminados (TER corto = 0.3 s, pausa = 8 s, ráfaga media = 5, RV-15, IV-60s).

**a)** Observa el panel de probabilidad relativa de reforzamiento (fila central). ¿La curva de RV es plana o tiene pendiente? ¿Qué significa una curva plana en términos de selección?

**b)** ¿La curva de IV sube o baja para TERs largos? ¿Cómo interpretas un valor de 3.0 en ese eje vertical?

**c)** En los paneles superiores, anota el porcentaje de TERs reforzados que son cortos bajo RV y bajo IV. ¿Cuánto difieren del porcentaje base de TERs cortos en la distribución general?

> *Clave:* El porcentaje base de TERs cortos con ráfaga media = 5 es aproximadamente (5−1)/5 = **80%**. ¿Los programas se alejan de ese valor? ¿En qué dirección y cuánto?

---

### Ejercicio 2 · Intermedio — Efecto del tamaño de ráfaga

Mantén RV-15 e IV-60s fijos. Prueba **ráfaga = 2**, **ráfaga = 5** y **ráfaga = 10**.

**a)** ¿Cómo cambia la distribución base de TERs (el histograma gris) cuando aumenta el tamaño de ráfaga? ¿Se vuelve más o menos bimodal?

**b)** ¿El porcentaje de TERs cortos reforzados bajo RV aumenta, disminuye o permanece igual cuando el tamaño de ráfaga aumenta? ¿Y bajo IV?

**c)** Con ráfaga = 2 (casi sin ráfagas, respuestas casi uniformemente distribuidas), ¿cómo cambia la diferencia entre RV e IV? ¿A qué se debe?

---

### Ejercicio 3 · Intermedio — El intervalo del IV

Mantén TER corto = 0.3 s, pausa = 8 s, ráfaga = 5, RV-15 fijos. Prueba IV con **20 s**, **60 s** y **180 s**.

**a)** Con IV-20s: ¿el reforzador llega más durante una ráfaga o durante una pausa? ¿Qué le pasa al porcentaje de TERs cortos reforzados?

**b)** Con IV-180s: el intervalo es mucho mayor que la pausa media (8 s). ¿El reforzador llega más frecuentemente al inicio de una ráfaga nueva o en otros momentos? ¿Por qué?

**c)** ¿Existe un valor del intervalo IV en que el porcentaje de TERs cortos reforzados se iguala al de RV? Predice si ese punto existe y qué pasaría con la evolución del comportamiento en ese caso.

---

### Ejercicio 4 · Avanzado — La evolución

Configura α = 0.15 y 25 bloques de entrenamiento. Observa el panel inferior izquierdo.

**a)** ¿RV o IV produce el cambio más rápido en la distribución de TERs? ¿A qué se debe esa diferencia en velocidad?

**b)** Aumenta α a 0.40. ¿El sistema se vuelve inestable u oscilante? ¿Por qué una tasa de aprendizaje muy alta puede ser contraproducente?

**c)** En el panel de desplazamiento (inferior derecho), describe el movimiento de las distribuciones de RV e IV a lo largo del entrenamiento. ¿Hacia dónde se desplaza cada una? ¿Qué predicción tiene esto para la tasa de respuesta observable en el registro acumulativo?

---

### Ejercicio 5 · Reflexión — Conexión con los datos empíricos

El capítulo describe la evidencia de Catania (1971): con la misma tasa de reforzamiento igualada, los organismos bajo RV responden 5 veces más rápido que bajo IV.

**a)** ¿El mecanismo mostrado en este simulador (reforzamiento diferencial de TERs) predice esa diferencia? ¿Por qué sí o por qué no?

**b)** El capítulo también menciona los programas DRL (*differential reinforcement of low rates*), que exigen que el TER exceda un mínimo para obtener el reforzador. ¿Dónde esperarías que cayera la curva de probabilidad relativa de reforzamiento para un DRL-20s? Dibuja mentalmente la forma de esa curva.

**c)** ¿Qué diferencia conceptual hay entre el efecto del reforzamiento diferencial de TERs (mecanismo molecular, mostrado aquí) y el efecto de la función de retroalimentación (mecanismo molar, Simulador 13.1)? ¿Son mecanismos alternativos o complementarios?


---

## Conexión teórica

Los patrones que genera este simulador ilustran los tres mecanismos analizados en el capítulo:

**1. Reforzamiento diferencial de tiempos entre respuestas (TER)**  
En RV, cada respuesta tiene la misma probabilidad de producir el reforzador, independientemente del TER que la precede — el programa es *neutro* respecto a la distribución de TER. En IV, la probabilidad de que un reforzador esté disponible crece con el tiempo transcurrido desde la última respuesta, por lo que el programa refuerza selectivamente los TER largos. La asimetría no es "RV refuerza TER cortos / IV refuerza TER largos", sino "RV es neutro / IV sesga hacia TER largos". Las tasas más bajas en IV resultan de ese sesgo unilateral.

**2. Sensibilidad a las correlaciones**  
El panel inferior muestra las funciones de retroalimentación del entorno: en RV, responder más *siempre* produce más reforzadores (función lineal). En IV, una vez alcanzada una tasa moderada, responder más no añade reforzadores (función cóncava con asíntota 1/t). Un organismo sensible a esta correlación responde más bajo razón.

**3. Función discriminativa del reforzador**  
En RF e IF, el reforzador señala que las próximas *n* respuestas (o los próximos *t* segundos) no producirán otro reforzador. El organismo aprende a usar el reforzador como señal temporal: de ahí la pausa post-reforzador en RF y el festoneo en IF.

> El panel inferior corresponde a la **Figura 13.5** del capítulo.  
> La curva de RV es lineal ($r = R/n$); la de IV es cóncava y asintótica ($r \to 1/t$ cuando $R \to \infty$).


---
## Créditos y licencia

Este notebook es parte del proyecto:

> **Bouzas, A. (2026).** *Aprendizaje y Comportamiento Adaptable: Principios y Modelos.*
> Lab25, Facultad de Psicología, UNAM.
> https://www.bouzaslab25.com

Apoyo en la construcción del simulador: **Eduardo Sánchez**.

Código disponible en: **https://github.com/bouzaslab25/libro-aca**
Licencia: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)
